## Retrieval-Augmented Generation (RAG) - Complete Practice Notebook (Google Colab)
Welcome to the consolidated RAG practice notebook. This file combines all ingestion, vector database, embedding, and evaluation exercises.

### <a id="table-of-contents"></a>📋 Table of Contents
* [Chapter 1: Introduction to RAG](#chapter-1-introduction-to-rag)
* [Chapter 2: Ingestion & Chunking](#chapter-2-ingestion-and-chunking)
* [Chapter 3: Vector Databases & Distance Metrics](#chapter-3-vector-databases-and-distance-metrics)
* [Chapter 4: RAG Evaluation & Advanced RAG](#chapter-4-rag-evaluation-and-advanced-rag)

### 🛠️ Setup Environment & Load API Keys

In [ ]:
# Install core RAG dependencies in Google Colab
!pip install -q langchain-core langchain-community langchain-google-genai langchain-groq chromadb python-dotenv pydantic

In [ ]:
import os
import getpass

# Set environment variables dynamically in Colab
if 'GROQ_API_KEY' not in os.environ:
    os.environ['GROQ_API_KEY'] = getpass.getpass('Enter your Groq API Key: ')
if 'GOOGLE_API_KEY' not in os.environ:
    os.environ['GOOGLE_API_KEY'] = getpass.getpass('Enter your Google API Key: ')

### <a id="chapter-1-introduction-to-rag"></a>📖 Chapter 1: Introduction to RAG
Retrieval-Augmented Generation (RAG) fetches relevant document context from an external database and appends it to the LLM query prompt.

#### 1.1 Simple In-Memory Mock RAG Setup

In [ ]:
# Basic in-memory document lookup
documents = {
    "python": "Python is a high-level programming language released in 1991.",
    "lcel": "LCEL is LangChain Expression Language used to compose chains."
}

def retrieve_document(query: str) -> str:
    for key, doc in documents.items():
        if key in query.lower():
            return doc
    return "No relevant document found."

query = "Tell me about python"
context = retrieve_document(query)
print("Retrieved Context:", context)

[Back to Top](#table-of-contents)

### <a id="chapter-2-ingestion-and-chunking"></a>📖 Chapter 2: Ingestion & Chunking

#### 2.1 CharacterTextSplitter Example

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

text = "Retrieval-Augmented Generation (RAG) is a technique for customizing LLM output. It uses external data."

splitter = CharacterTextSplitter(chunk_size=50, chunk_overlap=10, separator=" ")
chunks = splitter.split_text(text)
for i, chunk in enumerate(chunks):
    print(f"Chunk {i}: {chunk}")

#### 2.2 RecursiveCharacterTextSplitter Example

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(chunk_size=40, chunk_overlap=5)
rec_chunks = recursive_splitter.split_text(text)
for i, chunk in enumerate(rec_chunks):
    print(f"Rec Chunk {i}: {chunk}")

[Back to Top](#table-of-contents)

### <a id="chapter-3-vector-databases-and-distance-metrics"></a>📖 Chapter 3: Vector Databases & Distance Metrics

#### 3.1 Initializing Google GenAI Embeddings

In [ ]:
from langchain_google_genai import GoogleGenAIEmbeddings

embeddings = GoogleGenAIEmbeddings(model="models/text-embedding-004")
vector = embeddings.embed_query("Python programming")
print("Vector Length:", len(vector))
print("First 3 Dimensions:", vector[:3])

#### 3.2 ChromaDB Local Database Setup & Querying

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

# Create small list of Document structures
docs = [
    Document(page_content="LangGraph is used to construct stateful multi-agent workflows.", metadata={"source": "langgraph_docs"}),
    Document(page_content="ChromaDB is a lightweight embedded vector database.", metadata={"source": "chroma_docs"})
]

# Initialize Chroma store in-memory
db = Chroma.from_documents(docs, embeddings)

# Perform similarity search query
query = "Explain stateful multi-agent frameworks"
results = db.similarity_search(query, k=1)
print("Search Result:", results[0].page_content)
print("Metadata:", results[0].metadata)

[Back to Top](#table-of-contents)

### <a id="chapter-4-rag-evaluation-and-advanced-rag"></a>📖 Chapter 4: RAG Evaluation & Advanced RAG

#### 4.1 End-to-End RAG Chain with LCEL

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain.chat_models import init_chat_model

# Set up retriever interface
retriever = db.as_retriever(search_kwargs={"k": 1})

# Define RAG Prompt
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "Answer the question based ONLY on the following context:\n{context}"),
    ("human", "Question: {question}")
])

llm = init_chat_model("gemini-2.5-flash", model_provider="google_genai")

# Helper function to format document list
def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

# Compose RAG pipeline using LCEL
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt_template
    | llm
    | StrOutputParser()
)

output = rag_chain.invoke("What is LangGraph used for?")
print("Final Answer:", output)